In [43]:
from spacy.lang.en import English 

nlp = English()

#Add a sentencizer pipeline
nlp.add_pipe("sentencizer")
# sentencizer breaks a group of sentences into individual sentences

# Create a document instance as an exemple
doc = nlp("This is a sentence. This is another sentence. This is a third sentence.")
print(type(doc))
assert len(list(doc.sents)) == 3

# Acces the sentences of the document
list(doc.sents)

<class 'spacy.tokens.doc.Doc'>


[This is a sentence., This is another sentence., This is a third sentence.]

In [44]:
import pickle
import os

data_dir = "../data"
file_path = os.path.join(data_dir, "pages_and_texts.pkl")

#load the pages_and_texts list
try:    
    with open(file_path, "rb") as f:
        pages_and_texts = pickle.load(f)
except Exception as e:
    print(f"Error loading file: {e}")

print(f"First page: {pages_and_texts[0]}")

First page: {'page_number': -41, 'page_char_count': 29, 'page_word_count': 4, 'page_sentence_count_raw': 1, 'page_token_count': 7.25, 'text': 'Human Nutrition: 2020 Edition'}


In [45]:
from tqdm.auto import tqdm

for page_dict in tqdm(pages_and_texts):
    # for every page, create a list with its the sentences
    # create a new key "sentences" and assign it the list of sentences
    page_dict["sentences"] = list(nlp(page_dict["text"]).sents)

    # make sure all sentences are strings
    page_dict["sentences"] = [str(sent) for sent in page_dict["sentences"]]

    page_dict["page_sentences_count_spacy"] = len(page_dict["sentences"])


  0%|          | 0/1208 [00:00<?, ?it/s]

In [46]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
print(df.describe().round(2))


       page_number  page_char_count  page_word_count  page_sentence_count_raw  \
count      1208.00          1208.00          1208.00                  1208.00   
mean        562.50          1148.00           198.30                     9.97   
std         348.86           560.38            95.76                     6.19   
min         -41.00             0.00             1.00                     1.00   
25%         260.75           762.00           134.00                     4.00   
50%         562.50          1231.50           214.50                    10.00   
75%         864.25          1603.50           271.00                    14.00   
max        1166.00          2308.00           429.00                    32.00   

       page_token_count  page_sentences_count_spacy  
count           1208.00                     1208.00  
mean             287.00                       10.32  
std              140.10                        6.30  
min                0.00                        0.00  


On average each of our pages has 10 sentences and an total average of 287 token per page.

So a group with 10 sentences will give plenty of room for the text to embedded by 'all-mpnet-base-v2' model which has a capacity of 384 tokens.

To split our sentences into chunks of 10 or less, we create a function which recursively breaks down into sublists of a specified size.

In [47]:
#split size
num_sentences_chunk_size = 10

#recursive function to split a list into desired sizes
def split_list(list, slice_size = int) -> list[list[str]]:
    """
    Recursively split a list into sublists of a specified size.

    For example, a list of 17 sentences will be split into 2 sublists of [10] and [7]
    """
    final_list = []
    for i in range(0, len(list), slice_size):
        final_list.append(list[i:i+slice_size])
    return final_list

for page in pages_and_texts:
    page["sentences_chunks"] = split_list(page["sentences"], num_sentences_chunk_size)
    page["num_chunks"] = len(page["sentences_chunks"])


### Splitting each chunk into its own item
Before: each chunk is within his page

After: each chunk is an own item referring to his page

Create a new list of dictionaires each containing a single chunk of sentences with relative information

In [48]:
import re

pages_and_chunks = []
for page in pages_and_texts:
    for sentence_chunk in page["sentences_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = page["page_number"]

        #Join the sentence together into a paragraph-like structure, aka a chunk(so they are a single strin)
        joined_sentence_chunk = "".join(sentence_chunk).replace("  "," ").strip() #every sentence in one string(no spaces)
        # ".A" -> ". A" for any full-stop/capital letter combo
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk)
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        #Get stats about the chunk
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4
        chunk_dict["chunk_sentence_count"] = len(list(nlp(joined_sentence_chunk).sents))

        pages_and_chunks.append(chunk_dict)
    
#How many chunks do we have?
print(len(pages_and_chunks))


1843


In [49]:
import random
#view a random sample
random.sample(pages_and_chunks, k=1)


[{'page_number': 605,
  'sentence_chunk': 'view it online here: http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=354 Phytochemicals | 605',
  'chunk_char_count': 98,
  'chunk_word_count': 9,
  'chunk_token_count': 24.5,
  'chunk_sentence_count': 1}]

##### Now we're broken our whole textbook into chunks of 10 sentences or less as well as the page number they came from

##### WE check that the size of each chunk is less than the input embedding dimension required by our embeddings model(mpnet 384)

In [50]:
df = pd.DataFrame(pages_and_chunks)
print(df.shape)
print(df.describe().round(2))
print(df.info())

(1843, 6)
       page_number  chunk_char_count  chunk_word_count  chunk_token_count  \
count      1843.00           1843.00           1843.00            1843.00   
mean        583.38            734.44            112.33             183.61   
std         347.79            447.54             71.22             111.89   
min         -41.00             12.00              3.00               3.00   
25%         280.50            315.00             44.00              78.75   
50%         586.00            746.00            114.00             186.50   
75%         890.00           1118.50            173.00             279.62   
max        1166.00           1831.00            297.00             457.75   

       chunk_sentence_count  
count               1843.00  
mean                   6.58  
std                    3.23  
min                    1.00  
25%                    4.00  
50%                    7.00  
75%                   10.00  
max                   11.00  
<class 'pandas.core.frame.

In [51]:
print(df.columns)

Index(['page_number', 'sentence_chunk', 'chunk_char_count', 'chunk_word_count',
       'chunk_token_count', 'chunk_sentence_count'],
      dtype='object')


In [52]:
min_token_lenth = 30
for row in df[df["chunk_token_count"] <= min_token_lenth].sample(5).iterrows():
    print(f'Chunk token count: {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"]} | Sentence count: {row[1]["chunk_sentence_count"]}')

Chunk token count: 20.5 | Text: PART XVI CHAPTER 16. PERFORMANCE NUTRITION Chapter 16. Performance Nutrition | 931 | Sentence count: 3
Chunk token count: 26.75 | Text: Image by Allison Calabrese / CC BY 4.0 Figure 9.13 Niacin Deficiency, Pellagra 566 | Water-Soluble Vitamins | Sentence count: 1
Chunk token count: 4.25 | Text: Introduction | 61 | Sentence count: 1
Chunk token count: 19.25 | Text: http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=463   870 | Introduction | Sentence count: 1
Chunk token count: 20.75 | Text: http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=283   Alcohol Metabolism | 441 | Sentence count: 1


##### Looks like many of these are headers and footers of different pages, which dont offer to much information.

##### We filter to include only the chunks with over 30 tokens in length

In [53]:
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_lenth].to_dict(orient="records")
print(type(pages_and_chunks_over_min_token_len))
print(pages_and_chunks_over_min_token_len[:2])
print(f"Number of chunks: {len(pages_and_chunks_over_min_token_len)}")

<class 'list'>
[{'page_number': -39, 'sentence_chunk': 'Human Nutrition: 2020 Edition UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM ALAN TITCHENAL, SKYLAR HARA, NOEMI ARCEO CAACBAY, WILLIAM MEINKE-LAU, YA-YUN YANG, MARIE KAINOA FIALKOWSKI REVILLA, JENNIFER DRAPER, GEMADY LANGFELDER, CHERYL GIBBY, CHYNA NICOLE CHUN, AND ALLISON CALABRESE', 'chunk_char_count': 308, 'chunk_word_count': 42, 'chunk_token_count': 77.0, 'chunk_sentence_count': 1}, {'page_number': -38, 'sentence_chunk': 'Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and Human Nutrition Program is licensed under a Creative Commons Attribution 4.0 International License, except where otherwise noted.', 'chunk_char_count': 210, 'chunk_word_count': 30, 'chunk_token_count': 52.5, 'chunk_sentence_count': 1}]
Number of chunks: 1680


In [54]:
%%time

#'all-mpnet-base-v2 take as input maxim 384 words and outputs a veector with 768 dimensions
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device="mps")

sentences = [
    "The sentence transformers library provides an easy an open-source way to create embeddings.",
    "Sentences can be embedded one by one or a list of strins",
    "Embeddings are one of the mst powerful concepts in machine learning!",
    "Learn to use embeddins well and you'll be well on your way to being an AI engineer."
]

#Sentece are encoded/embedded by calling model.encode()
embeddings = embedding_model.encode(sentences)
embeddings_dict = dict(zip(sentences, embeddings))


CPU times: user 313 ms, sys: 1.96 s, total: 2.27 s
Wall time: 14.8 s


In [55]:
print(f"Shape of the embeddings: {embeddings.shape}")
print(f"Type of the embeddings: {type(embeddings)}")
list_embeddings = embeddings.tolist()
print(len(list_embeddings))
print(f"Type of the list_embeddings: {type(list_embeddings)}")
print(f"First embedding: {list_embeddings[0]}")
print(f"All embeddings: {list_embeddings}")
print(f"VERSIUNEA 2: {embeddings.shape}")

Shape of the embeddings: (4, 768)
Type of the embeddings: <class 'numpy.ndarray'>
4
Type of the list_embeddings: <class 'list'>
First embedding: [-0.02233651839196682, 0.04423441365361214, -0.015043305233120918, 0.06527683883905411, -0.023078221827745438, -0.013848783448338509, -0.004055370111018419, -0.05581676587462425, 0.01210063137114048, -0.02944711409509182, 0.031312208622694016, 0.04237889125943184, -0.0571415051817894, 0.029488923028111458, 0.032970257103443146, -0.04691527038812637, 0.04382812976837158, -0.0005009755841456354, -0.013974432833492756, 0.012750750407576561, 0.04760059714317322, 0.042417094111442566, 0.01972685009241104, 0.05109395831823349, -0.016120288521051407, -0.03397924080491066, 0.004096693824976683, -0.026752684265375137, 0.04522285237908363, 0.001244490733370185, -0.012511886656284332, -0.005988208577036858, 0.037972740828990936, 0.02167392149567604, 8.683613259563572e-07, -0.007547740824520588, -0.021511919796466827, 0.002461372409015894, 0.0069327964447

In [56]:
%%time

for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])

print(pages_and_chunks_over_min_token_len[0]["embedding"])

  0%|          | 0/1680 [00:00<?, ?it/s]

[ 6.74242303e-02  9.02282074e-02 -5.09549771e-03 -3.17545831e-02
  7.39082322e-02  3.51976305e-02 -1.97986625e-02  4.67691906e-02
  5.35726734e-02  5.01229428e-03  3.33928801e-02 -1.62216206e-03
  1.76080745e-02  3.62653248e-02 -3.16640479e-04 -1.07117631e-02
  1.54257566e-02  2.62176581e-02  2.77661136e-03  3.64942625e-02
 -4.44109328e-02  1.89362243e-02  4.90117818e-02  1.64020434e-02
 -4.85782959e-02  3.18294251e-03  2.72992942e-02 -2.04754574e-03
 -1.22828772e-02 -7.28048980e-02  1.20446226e-02  1.07300421e-02
  2.10002228e-03 -8.17773417e-02  2.67830205e-06 -1.81428511e-02
 -1.20803164e-02  2.47174725e-02 -6.27467260e-02  7.35438094e-02
  2.21624803e-02 -3.28767672e-02 -1.80095695e-02  2.22952347e-02
  5.61365038e-02  1.79514487e-03  5.25931641e-02 -3.31744994e-03
 -8.33882112e-03 -1.06284758e-02  2.31918227e-03 -2.23934669e-02
 -1.53011763e-02 -9.93053615e-03  4.65322584e-02  3.57468724e-02
 -2.54760236e-02  2.63694450e-02  3.74913891e-03 -3.82680260e-02
  2.58325636e-02  4.12872

In [57]:
print(len(pages_and_chunks_over_min_token_len[10]["sentence_chunk"]))
print(pages_and_chunks_over_min_token_len[10]["sentence_chunk"])
print(pages_and_chunks_over_min_token_len[10]["chunk_sentence_count"])


937
Part VII. Chapter 7. Alcohol Introduction University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 431 Alcohol Metabolism University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 436 Health Consequences of Alcohol Abuse University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 442 Health Benefits of Moderate Alcohol Intake University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 448 Part VIII. Chapter 8. Energy Introduction University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 453 The Atom University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 460 Weight Management University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 472
5


#### Batches - Computing on multiple samples at once

We dont embed each sentence, we take groups of sentences(batches) and embedd them together.

GPU has memory limitation, thats why Batch processing plays an important role



In [58]:
#turn text chunks into a single list
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]
print(type(pages_and_chunks_over_min_token_len[0]["sentence_chunk"]))

<class 'str'>


In [59]:
%%time

# Embed all texts in batches
text_chunk_embeddings = embedding_model.encode(
    text_chunks,
    batch_size = 32 #can use different batch sizes
)

CPU times: user 7.32 s, sys: 896 ms, total: 8.21 s
Wall time: 22.4 s


In [60]:
print(text_chunk_embeddings)
print(type(text_chunk_embeddings.tolist()))
print(text_chunk_embeddings.shape)
print(text_chunk_embeddings[0])
print(text_chunk_embeddings[0].shape)
print(text_chunk_embeddings[0].dtype)



[[ 0.06742424  0.09022819 -0.00509552 ... -0.02211543 -0.02321366
   0.01256908]
 [ 0.05521563  0.05921399 -0.01661672 ... -0.01204063 -0.01028474
   0.02273966]
 [ 0.02798017  0.03398143 -0.02064265 ... -0.00536188  0.02125601
   0.03130549]
 ...
 [ 0.07705157  0.00978555 -0.01218174 ... -0.04086807 -0.07517629
  -0.02405258]
 [ 0.10304513 -0.01647017  0.00826847 ... -0.05742175 -0.02828025
  -0.0294686 ]
 [ 0.08637737 -0.01253588 -0.01127466 ... -0.05223799 -0.03367288
  -0.02986604]]
<class 'list'>
(1680, 768)
[ 6.74242377e-02  9.02281925e-02 -5.09551819e-03 -3.17545347e-02
  7.39082024e-02  3.51976268e-02 -1.97986756e-02  4.67692055e-02
  5.35726845e-02  5.01230732e-03  3.33929136e-02 -1.62220665e-03
  1.76080503e-02  3.62653285e-02 -3.16682388e-04 -1.07117500e-02
  1.54257454e-02  2.62176488e-02  2.77660810e-03  3.64942253e-02
 -4.44109663e-02  1.89362131e-02  4.90117669e-02  1.64020434e-02
 -4.85782623e-02  3.18296184e-03  2.72992719e-02 -2.04757554e-03
 -1.22828986e-02 -7.280489

In [61]:
#Save embeedings to file
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)

embeddings_df_save_path = "../data/text_chunks_embeddings_df.csv"

text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [62]:
import random
import pandas as pd
import torch
import numpy as np

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
#import text adn embeddings df
text_chunks_and_embeddings_df = pd.read_csv(embeddings_df_save_path)
# #convert the embeddings back to np.array (it got converted to string when it got saved to csv)
text_chunks_and_embeddings_df["embedding"] = text_chunks_and_embeddings_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=' ', dtype=np.float32 ))

# #Convert text and embeddings df to a list of dicts
pages_and_chunks = text_chunks_and_embeddings_df.to_dict(orient="records")

# #Convert embeedings to tensors and send them to device(note: Numpy arrays are float64, torch tensors are float32)
embeddings = torch.tensor(np.array(text_chunks_and_embeddings_df["embedding"].tolist()), dtype = torch.float32).to(device)
embeddings.shape

Using device: mps


torch.Size([1680, 768])

In [63]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device="mps")

## R - Retrieval Part

In [64]:
from sentence_transformers import util
from time import perf_counter as timer

#Create a query
query = "macronutrients functions"
print(f"Our query: {query}")

#Encode the query
query_embedding = embedding_model.encode(query, convert_to_tensor=True)

start_time = timer()
#.dot_score returns a tensor of shape (1,n) - n dimensions(matrix)  -> we need to take the first element
dot_scores = util.dot_score(a=query_embedding, b=embeddings)[0]
print(f"[INFO] Dot product shape: {dot_scores}")

end_time = timer()

print(f'Time to get scores on {len(embeddings)} embeddings: {end_time - start_time} seconds')

#Get top k results
top_results_dot_product = torch.topk(dot_scores, k = 5)
top_results_dot_product

Our query: macronutrients functions
[INFO] Dot product shape: tensor([0.4343, 0.4406, 0.3667,  ..., 0.3941, 0.3321, 0.3707], device='mps:0')
Time to get scores on 1680 embeddings: 0.016599333001067862 seconds


torch.return_types.topk(
values=tensor([0.6926, 0.6738, 0.6646, 0.6536, 0.6473], device='mps:0'),
indices=tensor([42, 47, 41, 51, 46], device='mps:0'))

In [65]:
rez1 = top_results_dot_product[0]
print(f'Type of rez1: {type(rez1)}')
print(rez1)
rez2 = top_results_dot_product[1]
print(f'Type of rez2: {type(rez2)}')
values,indices = top_results_dot_product

zipped_results = zip(values.tolist(), indices.tolist())
print(type(zipped_results))
print(zipped_results)
zipped_results = list(zipped_results)
print(type(zipped_results))
print(zipped_results)


Type of rez1: <class 'torch.Tensor'>
tensor([0.6926, 0.6738, 0.6646, 0.6536, 0.6473], device='mps:0')
Type of rez2: <class 'torch.Tensor'>
<class 'zip'>
<class 'list'>
[(0.6925809383392334, 42), (0.6738271713256836, 47), (0.6646262407302856, 41), (0.6536345481872559, 51), (0.6472819447517395, 46)]


In [66]:
#helper function to print wrapped text
import textwrap

def print_wrapped(text, wrap_length = 80):
    wrapped_text = textwrap.fill(text, wrap_length)
    print(wrapped_text)

Now we can loop through the top_results_dot_products tuple and match up the scores and indicies and then use those indicies to index on our pages_and_chunks variable to get the relevant text chunk.

In [67]:
print(f"Query: '{query}'\n")
print(f"Results:")

for score, idx in zip(top_results_dot_product[0], top_results_dot_product[1]):
    # Print relevant sentence chunk (since the scores are in descending order, the most relevant chunk will be first)
    print(f"Score: {score:.4f}")
    print("Text:")
    print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
    #Print the page number too se we can reference the textbook further (and check the results)
    print(f"Page number: {pages_and_chunks[idx]['page_number']}")
    print("\n")
    

Query: 'macronutrients functions'

Results:
Score: 0.6926
Text:
Macronutrients Nutrients that are needed in large amounts are called
macronutrients. There are three classes of macronutrients: carbohydrates,
lipids, and proteins. These can be metabolically processed into cellular energy.
The energy from macronutrients comes from their chemical bonds. This chemical
energy is converted into cellular energy that is then utilized to perform work,
allowing our bodies to conduct their basic functions. A unit of measurement of
food energy is the calorie. On nutrition food labels the amount given for
“calories” is actually equivalent to each calorie multiplied by one thousand. A
kilocalorie (one thousand calories, denoted with a small “c”) is synonymous with
the “Calorie” (with a capital “C”) on nutrition food labels. Water is also a
macronutrient in the sense that you require a large amount of it, but unlike the
other macronutrients, it does not yield calories. Carbohydrates Carbohydrates
are 

In [68]:
def dot_product(vector1, vector2):
    return torch.dot(vector1, vector2)

def cosine_similarity(vector1, vector2):
    dot_product = torch.dot(vector1, vector2)

    #Get normalization for each vector(removes the magnitude, keeps direction)
    norm_vector1 = torch.sqrt(torch.sum(vector1 ** 2))
    norm_vector2 = torch.sqrt(torch.sum(vector2 ** 2))

    #Compute cosine similarity
    cosine_similarity = dot_product / (norm_vector1 * norm_vector2)
    return cosine_similarity
    

#Example tensors
vector1 = torch.tensor([1,2,3], dtype=torch.float32)
vector2 = torch.tensor([1,2,3], dtype=torch.float32)
vector3 = torch.tensor([4,5,6], dtype=torch.float32)
vector4 = torch.tensor([-1,-2,-3], dtype=torch.float32)

#Calculate dot product
print("Dot product between vector1 and vector2: ", dot_product(vector1, vector2))
print("Dot product between vector1 and vector3: ", dot_product(vector1, vector3))
print("Dot product between vector1 and vector4: ", dot_product(vector1, vector4))

#Calculate cosine similarity
print("Cosine similarity between vector1 and vector2: ", cosine_similarity(vector1, vector2))
print("Cosine similarity between vector1 and vector3: ", cosine_similarity(vector1, vector3))
print("Cosine similarity between vector1 and vector4: ", cosine_similarity(vector1, vector4))

Dot product between vector1 and vector2:  tensor(14.)
Dot product between vector1 and vector3:  tensor(32.)
Dot product between vector1 and vector4:  tensor(-14.)
Cosine similarity between vector1 and vector2:  tensor(1.0000)
Cosine similarity between vector1 and vector3:  tensor(0.9746)
Cosine similarity between vector1 and vector4:  tensor(-1.0000)


#### Functionizing our Semantic search Pipeline


In [69]:
def retrieve_relevant_sources(query: str, embeddings: torch.Tensor, model: SentenceTransformer = embedding_model, top_k: int = 5):
    """
    Retrieve the top k most relevant sources based on the query and embeddings.
    """

    # Embed the query
    query = embedding_model.encode(query, convert_to_tensor=True)

    #Get dot product
    start_time = timer()
    dot_scores = util.dot_score(a=query, b=embeddings)[0]
    end_time = timer()

    print(f"[INFO] Time to get scores on {len(embeddings)} embeddings: {end_time - start_time} seconds")
    scores, indices = torch.topk(dot_scores, k=top_k)

    return scores, indices

    #Get top k results
    
def print_top_result_and_score(query: str, embeddings: torch.Tensor, pages_and_chunks: list[dict],top_k: int = 5):

    scores, indices = retrieve_relevant_sources(query, embeddings, top_k=top_k)

    print(f"Query: '{query}'\n")
    print(f"Results:")

    for score, idx in zip(scores, indices):
        print(f"Score: {score:.4f}")
        print("Text:")
        print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
        print(f"Page number: {pages_and_chunks[idx]['page_number']}")
        print("\n")


In [70]:
query = "symptoms of pellagra"

print_top_result_and_score(query, embeddings, pages_and_chunks)

[INFO] Time to get scores on 1680 embeddings: 8.016599167604e-05 seconds
Query: 'symptoms of pellagra'

Results:
Score: 0.5000
Text:
Niacin deficiency is commonly known as pellagra and the symptoms include
fatigue, decreased appetite, and indigestion.  These symptoms are then commonly
followed by the four D’s: diarrhea, dermatitis, dementia, and sometimes death.
Figure 9.12  Conversion of Tryptophan to Niacin Water-Soluble Vitamins | 565
Page number: 565


Score: 0.3741
Text:
car. Does it drive faster with a half-tank of gas or a full one?It does not
matter; the car drives just as fast as long as it has gas. Similarly, depletion
of B vitamins will cause problems in energy metabolism, but having more than is
required to run metabolism does not speed it up. Buyers of B-vitamin supplements
beware; B vitamins are not stored in the body and all excess will be flushed
down the toilet along with the extra money spent. B vitamins are naturally
present in numerous foods, and many other foods ar

### G - Generation Part

In [71]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import is_flash_attn_2_available

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
#1. Pick a model
model_id = "google/gemma-2b-it"
print(f"[INFO] Loading model: {model_id}...")

#2. Instantiate tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

#3. Instantiate the model
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype = torch.float16,
    low_cpu_mem_usage = False, #use full memory
).to(device)

[INFO] Loading model: google/gemma-2b-it...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [72]:
def get_model_num_parameters(model: torch.nn.Module) -> int:

    return sum([p.numel() for p in model.parameters()])

get_model_num_parameters(llm_model)

2506172416

In [73]:
def get_model_mem_size(model: torch.nn.Module):
    """
    Get how much memory a pytorch model takes up
    """

    mem_params = sum([param.nelement() * param.element_size() for param in model.parameters()])
    mem_buffers = sum([buf.nelement() * buf.element_size() for buf in model.buffers()])

    #Calculate various model sizes
    model_mem_bytes = mem_params + mem_buffers #in bytes
    model_mem_mb = model_mem_bytes / (1024 ** 2) #in MB
    model_mem_gb = model_mem_bytes / (1024 ** 3) #in GB

    return {
        "model_mem_bytes": model_mem_bytes,
        "model_mem_mb": round(model_mem_mb, 2),
        "model_mem_gb": round(model_mem_gb, 2)
    }

get_model_mem_size(llm_model)

{'model_mem_bytes': 5012345344, 'model_mem_mb': 4780.15, 'model_mem_gb': 4.67}

##### Generating a text with our LLM

In [74]:
input_text = "What are the macronutrients, and what roles do they play in the human body?"
print(f'Input text: {input_text}')

#Create prompt template for instruction-tuned model
dialogue_template = [
    {"role": "user",
     "content":input_text}
]

# Apply the chat template
prompt = tokenizer.apply_chat_template(conversation=dialogue_template,
                                       tokenize=False, #keep as raw text(not tokenized)
                                       add_generation_prompt=True,
                                       add_special_tokens=True)

print(f'\nPrompt: {prompt}')

Input text: What are the macronutrients, and what roles do they play in the human body?

Prompt: <bos><start_of_turn>user
What are the macronutrients, and what roles do they play in the human body?<end_of_turn>
<start_of_turn>model



In [75]:
%%time

# Tokenize the input text (turn in into numbers) and send it to GPU
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
print(f'Model input (tokenized):\n {input_ids}\n')

# Generate outputs passed on the tokenized input
outputs = llm_model.generate(**input_ids, max_new_tokens = 256) #maxim number of tokens to create

print(f"Model output (tokens): {outputs[0]}")

Model input (tokenized):
 {'input_ids': tensor([[     2,      2,    106,   1645,    108,   1841,    708,    573, 186809,
         184592, 235269,    578,   1212,  16065,    749,    984,   1554,    575,
            573,   3515,   2971, 235336,    107,    108,    106,   2516,    108]],
       device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]], device='mps:0')}

Model output (tokens): tensor([     2,      2,    106,   1645,    108,   1841,    708,    573, 186809,
        184592, 235269,    578,   1212,  16065,    749,    984,   1554,    575,
           573,   3515,   2971, 235336,    107,    108,    106,   2516,    108,
         21404, 235269,   1517, 235303, 235256,    476,  25497,    576,    573,
        186809, 184592,    578,   1024,  16065,    575,    573,   3515,   2971,
        235292,    109,    688,  12298,   1695, 184592,  66058,    109, 235287,
          5231, 156615,  56227,  66058,    108,    

In [76]:
#Decode the output tokens to text
outputs_decoded = tokenizer.decode(outputs[0])
print(f"Model output (decoded): {outputs_decoded}")

Model output (decoded): <bos><bos><start_of_turn>user
What are the macronutrients, and what roles do they play in the human body?<end_of_turn>
<start_of_turn>model
Sure, here's a breakdown of the macronutrients and their roles in the human body:

**Macronutrients:**

* **Carbohydrates:**
    * Provide energy for the body's cells and tissues.
    * Carbohydrates are the primary source of energy for most cells.
    * Complex carbohydrates are those that take longer to digest, such as whole grains, fruits, and vegetables.
    * Simple carbohydrates are those that are quickly digested, such as sugar, starch, and lactose.

* **Proteins:**
    * Build and repair tissues, enzymes, and hormones.
    * Proteins are essential for immune function, hormone production, and tissue repair.
    * There are different types of proteins, each with specific functions.

* **Fats:**
    * Provide energy, insulation, and help absorb vitamins.
    * Healthy fats include olive oil, avocado, nuts, and seeds.
  

In [77]:
print(f"Input text: {input_text}")
print(f"Output text: {outputs_decoded.replace(prompt, '').replace('<bos>', '').replace('<eos>', '')}")


Input text: What are the macronutrients, and what roles do they play in the human body?
Output text: Sure, here's a breakdown of the macronutrients and their roles in the human body:

**Macronutrients:**

* **Carbohydrates:**
    * Provide energy for the body's cells and tissues.
    * Carbohydrates are the primary source of energy for most cells.
    * Complex carbohydrates are those that take longer to digest, such as whole grains, fruits, and vegetables.
    * Simple carbohydrates are those that are quickly digested, such as sugar, starch, and lactose.

* **Proteins:**
    * Build and repair tissues, enzymes, and hormones.
    * Proteins are essential for immune function, hormone production, and tissue repair.
    * There are different types of proteins, each with specific functions.

* **Fats:**
    * Provide energy, insulation, and help absorb vitamins.
    * Healthy fats include olive oil, avocado, nuts, and seeds.
    * Trans fats can raise cholesterol levels and increase the ri

### A - Augmenting Generation with Retrival

In [78]:
def prompt_formatter(query: str, context_items: list) -> str:
    """
    Augement the query with the text-based context from context_items
    """
    #join context items into a single dotted paragraph
    context = "- " + "\n- ".join([item["sentence_chunk"] for item in context_items])
    
    #model: gemma-2b-it -> 'it' stands for instruction-tuned
    #we create a base prompt with example to help the model
    base_prompt = """Base on the following context items, please answer the query.
    Give yourself room to think by extraxting relevant passages from the context before answering the query. Don't return the thinking, only return the answer.
    Make sure yout answers are as explanatory as possible.
    Use the following examples as refernce for the ideal answer style.
    \nExample 1:
    Query: What are the fat-soluble vitamins?
    Answer: The fat-soluble vitamins include Vitamin A, D, E, and K. These vitamins are absorbed through the fat-rich diet and stored in the liver and fat tissues.
    \nExample 2:
    Query: What are the causes of type 2 diabetes?
    Answer: Type 2 diabetes is primarily caused by a combination of genetic factors and lifestyle factors such as obesity and physical inactivity.
    \nNow use the following context items to answer the user query:
    {context}. The context contains the answer. Look carefully and extract relevant information.
    \nRelevant passages: <extract relevant passages from the context here>
    User query: {query}
    Answer:"""

    #Update base prompt with context items and query
    base_prompt = base_prompt.format(context = context, query = query)

    dialogue_template = [
        {"role":"user",
         "content":base_prompt}
    ]
    print(f"Base prompt: {base_prompt}")
    #Apply the chat template
    prompt = tokenizer.apply_chat_template(conversation=dialogue_template,
                                          tokenize=False,
                                          add_generation_prompt=True)
    
    return prompt
    

In [79]:
query = "What are the fat-soluble vitamins?"
print(f"Query: {query}")

#Get relevant resources
scores, indices = retrieve_relevant_sources(query, embeddings)

#Create a list of context items
context_items = [pages_and_chunks[i] for i in indices]

#Format prompt with context items
prompt = prompt_formatter(query, context_items)

print(prompt)


Query: What are the fat-soluble vitamins?
[INFO] Time to get scores on 1680 embeddings: 0.0002474999928381294 seconds
Base prompt: Base on the following context items, please answer the query.
    Give yourself room to think by extraxting relevant passages from the context before answering the query. Don't return the thinking, only return the answer.
    Make sure yout answers are as explanatory as possible.
    Use the following examples as refernce for the ideal answer style.
    
Example 1:
    Query: What are the fat-soluble vitamins?
    Answer: The fat-soluble vitamins include Vitamin A, D, E, and K. These vitamins are absorbed through the fat-rich diet and stored in the liver and fat tissues.
    
Example 2:
    Query: What are the causes of type 2 diabetes?
    Answer: Type 2 diabetes is primarily caused by a combination of genetic factors and lifestyle factors such as obesity and physical inactivity.
    
Now use the following context items to answer the user query:
    - Jour

#### Our prompt is ready to be tokenized and passed to our LLM

In [80]:
%%time

input_ids = tokenizer(prompt, return_tensors="pt").to(device)

#Generate outputs
outputs = llm_model.generate(**input_ids,
                            max_new_tokens = 256,
                            temperature = 0.7, #lower temperature -> more deterministic(less random)
                            do_sample = True,)

#Decode the output tokens to text
outputs_text = tokenizer.decode(outputs[0])

print(f"Input text: {input_text}")
print(f"Output text: {outputs_text.replace(prompt, '')}")

Input text: What are the macronutrients, and what roles do they play in the human body?
Output text: <bos>Sure, here's the answer to the user's query:

The context mentions that fat-soluble vitamins are vitamins that are absorbed through the fat-rich diet and stored in the liver and fat tissues. They are mainly found in foods containing fat, and some fat-soluble vitamins (such as vitamin A) are also found in naturally fat-free foods such as green leafy vegetables, carrots, and broccoli.<eos>
CPU times: user 17 s, sys: 766 ms, total: 17.8 s
Wall time: 19 s


### Entire RAG Pipeline in one function

In [81]:
def ask(query: str,
        temperature: 0.7,
        max_new_tokens=512,
        format_answer_text=True,
        return_answer_only=True,
        ):
    """
    Takes a query, finds relevant resources/context and generates an answer to the query based on the relevant resources
    """

    # Get just the scores and indices of the related results
    scores, indices = retrieve_relevant_sources(query, embeddings)

    #Create a list of context items
    context_items = [pages_and_chunks[i] for i in indices]

    #add score to context items
    for i, item in enumerate(context_items):
        item["score"] = scores[i]

    #Format prompt with context items
    prompt = prompt_formatter(query, context_items)

    #Tokenize the prompt
    input_ids = tokenizer(prompt, return_tensors="pt").to(device)

    #Generate an output of tokens
    outputs = llm_model.generate(**input_ids,
                                max_new_tokens = max_new_tokens,
                                temperature = temperature,
                                do_sample = True)
    output_text = tokenizer.decode(outputs[0])

    #Format the output text
    if format_answer_text:
        #Replace special tokens and unecessary help message
        output_text = output_text.replace(prompt, "").replace("<bos>", "").replace("<eos>", "").replace("Sure, here is the answer to the user queery: \n\n", "")

    #Return the answer only if requested
    if return_answer_only:
        return output_text
    
    return output_text, context_items

In [82]:
query = "What are the fat-soluble vitamins?"

answer, context_items = ask(query, temperature=0.7, max_new_tokens=512, return_answer_only=False)

print(f"Answer: {answer}")
print_wrapped(answer)
print(f"Context items: {context_items}")

[INFO] Time to get scores on 1680 embeddings: 7.762500899843872e-05 seconds
Base prompt: Base on the following context items, please answer the query.
    Give yourself room to think by extraxting relevant passages from the context before answering the query. Don't return the thinking, only return the answer.
    Make sure yout answers are as explanatory as possible.
    Use the following examples as refernce for the ideal answer style.
    
Example 1:
    Query: What are the fat-soluble vitamins?
    Answer: The fat-soluble vitamins include Vitamin A, D, E, and K. These vitamins are absorbed through the fat-rich diet and stored in the liver and fat tissues.
    
Example 2:
    Query: What are the causes of type 2 diabetes?
    Answer: Type 2 diabetes is primarily caused by a combination of genetic factors and lifestyle factors such as obesity and physical inactivity.
    
Now use the following context items to answer the user query:
    - Journal of the National Cancer Institute, 96(2

In [83]:
# Load the model
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B").to(device)
print(f"Using device: {device}")

Using device: mps


In [84]:
query = "What are the fat-soluble vitamins?"
print(f"Query: {query}")
#Encode the query
query_embedding = model.encode(query, convert_to_tensor=True)

#Get dot product

    
    

Query: What are the fat-soluble vitamins?


In [85]:
copy = pages_and_chunks
print(type(copy))
print(type(copy[0]))
print(copy[0].keys())
start_time = timer()
batch_size = 100
for item in copy:
    item["embeddings_qwen"] = model.encode(item["sentence_chunk"])
end_time = timer()
print(f"Time to embed {len(copy)} sentences: {end_time - start_time} seconds")

<class 'list'>
<class 'dict'>
dict_keys(['page_number', 'sentence_chunk', 'chunk_char_count', 'chunk_word_count', 'chunk_token_count', 'chunk_sentence_count', 'embedding'])


KeyboardInterrupt: 

In [86]:
sentence_list = [item["sentence_chunk"] for item in copy]
print(type(sentence_list))
print(f"Length of sentence_list: {len(sentence_list)}")
print(f"First sentence: {sentence_list[0]}")
start_time = timer()
batch_size = 100
for i in range(0, len(sentence_list), batch_size):
    batch = sentence_list[i:i+batch_size]
    embeddings_batch = model.encode(batch)

end_time = timer()
print(f"Time to embed {len(copy)} sentences: {end_time - start_time} seconds")

<class 'list'>
Length of sentence_list: 1680
First sentence: Human Nutrition: 2020 Edition UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM ALAN TITCHENAL, SKYLAR HARA, NOEMI ARCEO CAACBAY, WILLIAM MEINKE-LAU, YA-YUN YANG, MARIE KAINOA FIALKOWSKI REVILLA, JENNIFER DRAPER, GEMADY LANGFELDER, CHERYL GIBBY, CHYNA NICOLE CHUN, AND ALLISON CALABRESE


KeyboardInterrupt: 

In [87]:
if torch.backends.mps.is_available():
    # Elibereaza toata memoria cached de MPS, facand-o disponibila pentru sistem.
    torch.mps.empty_cache()
    print("Memoria cache MPS a fost golită.")
else:
    print("MPS nu este disponibil.")

Memoria cache MPS a fost golită.


In [ ]:
copy = pages_and_chunks
sentence_list = [item["sentence_chunk"] for item in copy]
start_time = timer()
embeddings_batch_v2 = model.encode(
        sentence_list,
        batch_size = 32
    )
end_time = timer()
print(f"Time to embed {len(copy)} sentences: {end_time - start_time} seconds")

Time to embed 1680 sentences: 803.6967788750771 seconds


In [88]:
print(type(copy[0]["embedding"]))
print(copy[0]["embedding"].shape)
print(type(copy[0]["embeddings_batch_v2"]))

<class 'numpy.ndarray'>
(768,)


KeyError: 'embeddings_batch_v2'

In [89]:
print(copy[0].keys())
print(copy[0]["embeddings_qwen"].dtype)
print(copy[0]["embeddings_qwen"].shape)

dict_keys(['page_number', 'sentence_chunk', 'chunk_char_count', 'chunk_word_count', 'chunk_token_count', 'chunk_sentence_count', 'embedding', 'embeddings_qwen'])
float32
(1024,)


In [ ]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    answer_relevancy,
    faithfulness
)

#Try to import aditional metrics if available
try:
    from ragas.metrics import context_entity_recall
except ImportError:
    context_entity_recall = None
    print("context_entity_recall metric not found")

try:
    from ragas.metrics import noise_robustness
except ImportError:
    noise_robustness = None
    print("noise_robustness metric not found")


noise_robustness metric not found


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
eval_questions = [
    "How often should infants be breastfed?",
    "What are symptoms of pellagra?",
    "How does saliva help with digestion?",
    "What is the recommended protein intake per day, based on you weight?",
    "What are micronutrients?"
]

ground_truth_answers = [
    "Infants should generally be breastfed 8–12 times in 24 hours during the first weeks of life, or about every 2–3 hours on demand. Feeding frequency gradually decreases as the baby grows, but exclusive breastfeeding is recommended for about the first 6 months",
    "Pellagra is caused by niacin (vitamin B3) deficiency and is classically described by the “4 Ds”: dermatitis, diarrhea, dementia, and if untreated, death. Other symptoms include inflamed skin (especially in sun‑exposed areas), mouth sores, weakness, confusion, and loss of appetite",
    "Saliva moistens and lubricates food, making it easier to chew and swallow. It contains enzymes like amylase and lipase that begin breaking down starches (and some fats) in the mouth. Saliva also helps with taste perception, maintains oral pH, protects teeth, and provides antimicrobial action",
    "The general guideline is 0.8 grams of protein per kilogram of body weight per day for sedentary adults. Active individuals may need 1.2–2.0 g/kg/day, and athletes or those aiming for muscle growth may go up to 2.2 g/kg/day. For example, a 70 kg adult needs about 56 g/day at minimum",
    "Micronutrients are vitamins and minerals that the body needs in small amounts for essential functions such as energy production, immune support, bone health, and growth. They include water‑soluble vitamins (B‑complex, C), fat‑soluble vitamins (A, D, E, K), macrominerals (calcium, magnesium, potassium), and trace minerals (iron, zinc, iodine, selenium)"
]

In [ ]:
def generate_answer(query):
    """Generate RAG answer for a given query"""
    scores, indices = retrieve_relevant_sources(query, embeddings)

    #Create a list of context items
    context_items = [pages_and_chunks[i] for i in indices]

    #Format prompt with context items
    prompt = prompt_formatter(query, context_items)

    #Generate answer
    input_ids = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = llm_model.generate(**input_ids,
                                temperature=0.7,
                                do_sample=True,
                                max_new_tokens=256)

    #Extract only the generated answer (remove the prompt)
    output_text = tokenizer.decode(outputs[0])
    answer = output_text.replace(prompt, "").strip()

    #Return both answer and context
    context = [item["sentence_chunk"] for item in context_items]
    return answer, context


In [ ]:
evaluation_data = []

print(f"Generating RAG answers for evaluation...")
for question, ground_truth in zip(eval_questions, ground_truth_answers):
    print(f"Processing question: {question[:50]}...")

    #Generate answer
    rag_answer, contexts = generate_answer(question)

    evaluation_data.append({
        "question": question,
        "ground_truth": ground_truth,
        "answer": rag_answer,
        "contexts": contexts
    })
    

Generating RAG answers for evaluation...
Processing question: How often should infants be breastfed?...
[INFO] Time to get scores on 1680 embeddings: 0.0009830419439822435 seconds
Processing question: What are symptoms of pellagra?...
[INFO] Time to get scores on 1680 embeddings: 0.00017200014553964138 seconds
Processing question: How does saliva help with digestion?...
[INFO] Time to get scores on 1680 embeddings: 9.504100307822227e-05 seconds
Processing question: What is the recommended protein intake per day, ba...
[INFO] Time to get scores on 1680 embeddings: 0.00026899995282292366 seconds
Processing question: What are micronutrients?...
[INFO] Time to get scores on 1680 embeddings: 0.00014845794066786766 seconds


In [ ]:
# Convert to a dataset format required by RAGAs
eval_dataset = Dataset.from_pandas(pd.DataFrame(evaluation_data))

metrics = [
    context_precision,
    context_recall,
    answer_relevancy,
    faithfulness
]

#Evaluate the RAG system
results = evaluate(eval_dataset,
                   llm = llm_model,
                   metrics = metrics)

#Print the results
results_df = results.to_pandas()

AttributeError: 'GemmaForCausalLM' object has no attribute 'set_run_config'

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk.


NameError: name 'pipeline' is not defined

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

system_prompt = """You are an expert chef and time management specialist.
You will be given a dish name and recipe. Analyze the recipe and estimate:
1. Total cooking time (in minutes)
2. Breakdown of time for each type of activity (prep, cooking, resting, etc.)
3. Difficulty level (Beginner/Intermediate/Advanced)

Return as JSON:
{
    "total_time": <total minutes>,
    "step_breakdown": {
        "prep": <minutes>,
        "cooking": <minutes>,
        "resting": <minutes if applicable>
    },
    "difficulty": "Beginner|Intermediate|Advanced"
}"""

time_template = ChatPromptTemplate([
    ("system", system_prompt),
    ("human", "Dish: {dish}\n\nRecipe: {recipe}")
])

def create_time_estimation_chain():
    time_chain = (
        {"dish": lambda x: x["dish"], "recipe": lambda x: x["recipe"]}
        | time_template
        | llm
        | JsonOutputParser()
    )
    return time_chain

# Integrează în lanțul final:
def create_final_chain_with_time():
    time_chain = create_time_estimation_chain()
    
    final_chain = (
        {"pantry": lambda x: x["pantry"]}
        | pantry_chain
        | {"ingredients": lambda x: str(x)}
        | recipe_chain
        | RunnableParallel({
            "allergen_info": allergen_chain,
            "time_info": time_chain
        })
    )
    return final_chain

NameError: name 'ChatPromptTemplate' is not defined